# 01. Setup, Chargement et Fusion

Initialiser Spark, charger les jeux bruts et construire le dataset train fusionné.

**Sortie principale :** `artifacts/01_setup_load_merge/data/01_train_merged.parquet`

In [4]:
# Importation des bibliothèques principales
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, round as spark_round

from pyspark.ml.stat import *
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import  VectorAssembler , RobustScaler
from pyspark.ml.feature import StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.classification import RandomForestClassifier,LogisticRegression,GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator ,MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

#  Visualisation et analyse complémentaire

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from builtins import round


In [5]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
RAW_DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_ROOT = PROJECT_DIR / "artifacts"
MODELS_ROOT = PROJECT_DIR / "models"
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

STAGE_NAME = "01_setup_load_merge"
STAGE_ARTIFACTS_DIR = ARTIFACTS_ROOT / STAGE_NAME
STAGE_DATA_DIR = STAGE_ARTIFACTS_DIR / "data"

for _dir in (ARTIFACTS_ROOT, MODELS_ROOT, STAGE_ARTIFACTS_DIR, STAGE_DATA_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

In [6]:
# Création de la session Spark

spark = SparkSession.builder \
    .appName("DetectionFraude") \
    .master("local[8]") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate() 

spark 

In [7]:
# fonciton nettoyage de mémoire 
def cleanup(*vars_to_del):
    for var in vars_to_del:
        del var
    import gc; gc.collect()
    spark.catalog.clearCache()

## 1. Chargement et fusion des données

###  Description des données

Le jeu de données est constitué de deux fichiers complémentaires :

####  `train_transaction.csv` — Transactions financières

Contient les données principales liées à chaque opération bancaire :
- `TransactionID` : identifiant unique de la transaction.
- `isFraud` : cible binaire (1 = fraude, 0 = normal).
- `TransactionAmt` : montant de la transaction.
- `ProductCD` : type de produit acheté.
- `card1` à `card6` : identifiants partiels de carte bancaire.
- `addr1`, `addr2` : zones géographiques.
- `dist1`, `dist2` : proximités géographiques.
- `P_emaildomain`, `R_emaildomain` : domaines email.
- `C1` à `C14` : agrégats comportementaux.
- `D1` à `D15` : deltas temporels (liés à la récence).
- `M1` à `M9` : indicateurs binaires (matchs device/adresse...).
- `V1` à `V339` : variables anonymisées dérivées internes.

####  `train_identity.csv` — Identité des utilisateurs

Contient des métadonnées liées aux appareils et aux profils :
- `id_01` à `id_38` : variables numériques et catégorielles décrivant l’identité technique, l’environnement de connexion et l’appareil.
- `DeviceType`, `DeviceInfo` : type et informations relatives à l’appareil utilisé.
- `id_30`, `id_31` : système d’exploitation et navigateur détectés.


###  Chargement des données avec PySpark

In [8]:
#  Chargement des datasets

df_transaction = spark.read.csv((RAW_DATA_DIR / "train_transaction.csv").as_posix(), header=True, inferSchema=True)
df_identity = spark.read.csv((RAW_DATA_DIR / "train_identity.csv").as_posix(), header=True, inferSchema=True)

# Dim datasets 

print(f"Lignes transaction : {df_transaction.count()}, Colonnes transaction : {len(df_transaction.columns)}")
print(f"Lignes identity    : {df_identity.count()}, Colonnes identity    : {len(df_identity.columns)}")


# Vérif des colonnes communes pour la jointure

colonnes_communes = set(df_transaction.columns).intersection(set(df_identity.columns))

print(f"Clés communes : {colonnes_communes}")

Lignes transaction : 590540, Colonnes transaction : 394
Lignes identity    : 144233, Colonnes identity    : 41
Clés communes : {'TransactionID'}


###  Fusion des jeux de données

#### Objectif :
Associer les données d’identité aux transactions, sans exclure celles qui n’en ont pas. Cela permet de garder un volume complet et de marquer la présence ou non d’identité.

####  Détails techniques :
- Jointure effectuée sur `TransactionID`
- Type de jointure : `LEFT JOIN`
- Dataset principal : `df_transaction`
- Dataset secondaire : `df_identity`

In [9]:
# Fusion des deux fichier train  

df = df_transaction.join(df_identity,on="TransactionID", how="left")

print(f"nb de ligne : {df.count()} et de  colonne : {len(df.columns)}")


nb de ligne : 590540 et de  colonne : 434


###  Synthèse post-fusion des données

Sur les **590 540 transactions**, seulement **144 233** ont des informations d'identité dans le fichier `identity.csv`

Cela signifie que **plus 446 000 des transactions** sont totalement **anonymes** dans les colonnes comme `DeviceType`, `id_01`, `id_30`, etc.

Cette absence d’identité **n’est pas une erreur**, mais un **signal potentiellement révélateur**. Un fraudeur pourrait chercher à **éviter d’être traçable** en ne fournissant pas d’identifiants techniques.


In [10]:
from fraud_pipeline import add_has_identity

df = add_has_identity(df)

output_path = STAGE_DATA_DIR / "01_train_merged.parquet"
df.write.mode("overwrite").parquet(output_path.as_posix())
print(f"Train fusionne sauvegarde dans {output_path}")

Train fusionne sauvegarde dans C:\Users\El-fahad COMBO\Desktop\Fraud_Detection_IEEE\artifacts\01_setup_load_merge\data\01_train_merged.parquet
